# Lab 01: The Reflection Pattern for Evidence-Bounded Reasoning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab01_reflection_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

## What this lab is about

In this lab an AI model plays the part of a **junior examiner**. It reads evidence taken from a
phone and writes a first **incident report**: a short written account of what seems to have
happened, backed by the evidence. The evidence comes as **artifacts**: the files and records
pulled from the phone.

Then a second AI, playing a **senior reviewer**, **critiques** the report. It points out claims
that go further than the evidence. Finally the first AI **revises** its report using that
critique.

Write a draft, review it, fix it: this three-move loop is called the **Reflection Pattern**.
"Reflection" simply means the AI's work is checked and improved before anyone relies on it.

Your job is **not** to trust the AI. Your job is to read each stage, trace every claim back to
the evidence, and decide whether the final report stays inside what the artifacts actually
show. The AI helps with the checking. The forensic judgment stays with you.

You will do the loop **by hand first**, one move at a time (Part A). Then you will run the
**same loop in one go** with a ready-made helper called `ReflectionAgent` (Part B).

## How to use this notebook

- A notebook is made of **cells**. Text cells (like this one) explain. Code cells (the grey
  boxes) do the work.
- Run the code cells **one at a time, from top to bottom**. Click a cell and press
  **Shift + Enter**, or click its Run button. Wait for each cell to finish before the next.
- Every code cell has a text cell just before it with four parts: **Purpose** (why the step
  exists), **Input** (what it uses, and which earlier step made it), **How it works** (what
  the code does, in plain words), and **Output** (what you will see, and what the next step
  uses).
- You do not need to know Python. The lines starting with `#` inside the code cells are
  comments: plain-English notes that explain what each part does. The computer ignores them.
- If the session restarts (for example after a long break), run the notebook again from the
  top.

## The plan at a glance

Each step hands something to the next one. This table shows the chain.

| Step | What happens | What it hands on |
|---|---|---|
| 0.1 to 0.3 | Setup: install one add-on, connect to the AI model, write the case files | `client`, `MODEL`, the `data/` folder |
| A.1 | Pack the evidence files into one block of text | `artifact_bundle` |
| A.2 | Write the analyst's instructions and task | `generation_chat_history` |
| A.3 | The AI analyst writes a first draft | `incident_report_draft` |
| A.4 | A separate AI reviewer critiques the draft | `critique` |
| A.5 | The analyst revises the draft using the critique | `revised_incident_report` |
| A.6 | Reading only: how the loop repeats | (nothing) |
| B.1 | Load the ready-made `ReflectionAgent` | `ReflectionAgent` |
| B.2 | The agent runs the whole loop by itself | `final_report` |
| B.3 | Show the final report | (nothing) |
| Your Evidence Check | You judge the AI's work | your `answers` |

> **Before this notebook:** if your instructor assigned the 35-minute group evidence activity,
> finish it first and keep your answers. You will compare them against the AI's output.

## Part 0: One-Time Setup

These three steps get the notebook ready. They install one small add-on, connect the notebook
to a **free AI model** that runs on the internet (on a service called Groq), and create the
case files. Everything runs in this hosted session; nothing is installed on your own computer.

### Step 0.1: Install the add-on that talks to the AI model

**Purpose:** add the one tool this notebook needs to send messages to an AI model and read its
replies.

**Input:** nothing from you. This is the first step.

**How it works:** the command `%pip install` downloads a small, free Python add-on (called a
**library**) named `openai` into this hosted session. Despite its name, it can talk to several
AI services, including Groq, which we use here.

**Output:** a few lines of install messages, or none at all. Warnings are normal and safe to
ignore. The add-on is now ready for Step 0.2.

In [ ]:
# Step 0.1: install the one add-on this notebook needs, inside this hosted session only
# (not on your own computer). The -q means "quiet": show fewer messages.
%pip install -q openai

### Step 0.2: Connect to a free AI model (Groq)

**Purpose:** connect the notebook to a free AI model, so the later steps can send it evidence
and get written answers back.

**Input:** the `openai` add-on from Step 0.1, and a free Groq **API key**. An API key is a
private password that lets the notebook use the AI model. Groq offers free keys and asks for
no credit card. To get one:

1. Open **https://console.groq.com/keys** and sign in (a Google or GitHub account works).
2. Click **Create API Key** and copy the key.
3. Run the cell below and paste the key when asked. It stays hidden while you type and is
   **not saved** into the notebook. Treat it like a password: do not share it.

**How it works:** the code keeps your key for this session only and connects to Groq. Then it
picks the first AI model on its short list that your key can use. It also adds a small safety
helper that keeps every request inside Groq's free limits. Finally it sends a one-word test
message to prove the connection works.

**Output:** a line such as `Model OK: openai/gpt-oss-120b -> ready`. The cell also creates two
things that every AI step below uses: `client` (the connection to Groq) and `MODEL` (the name
of the AI model it chose).

In [ ]:
# Step 0.2: connect this notebook to a free AI model hosted by Groq.
import os        # os: lets the notebook read and store settings, such as your key
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: sends your messages to the AI model and brings back replies
                          # (Groq understands the same message format)

# Ask for your key only if this session does not already have one stored.
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

# The web address of Groq's service. Every message to the AI model is sent here.
BASE_URL = 'https://api.groq.com/openai/v1'
# A short list of free AI models that work well for these labs, best first. Groq retires models
# from time to time, so this cell uses the first one your key can reach and prints its name.
# You can edit this list to try a different model.
PREFERRED = ['openai/gpt-oss-120b', 'qwen/qwen3.8-27b']

# Build the 'client': the connection that every later step uses to talk to the AI model.
# The free plan only allows a certain amount of text per minute (about 8,000 tokens; a token is
# a small piece of text, roughly three-quarters of a word). If you go over, Groq says 'try again
# in a few seconds'. max_retries=8 tells the client to wait and try again instead of failing.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# Every AI request in this notebook passes through the small helper below. You do not need to
# follow every line. It does two things:
# 1. It caps how long each reply may be: 1,500 tokens, or 900 for the Qwen model, whose free
#    plan allows only 1,000 reply tokens a minute. Groq counts your request PLUS the longest
#    possible reply against your per-minute allowance, so a cap keeps you inside it.
#    (Any single request can still set its own max_tokens.)
# 2. The gpt-oss model sometimes tries to use a built-in feature instead of writing plain text,
#    and Groq refuses that ('model called a tool'). The helper then simply asks again, adding
#    a reminder to reply in plain text. It tries at most 3 times.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
client.chat.completions.create = _create_within_limits

# Ask Groq which models your key can use, then pick the first one from PREFERRED that is there.
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
# If none of our preferred models is available, list the ones that are, and stop here so you
# can put one of them into PREFERRED.
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

# Send a tiny test message to prove the whole connection works.
probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the case files

**Purpose:** create the evidence for case `DF-2026-017` so you can examine it. The six files
are **synthetic**: made up for teaching, so no real person's data is involved. The same files
are also kept in the course repository.

**Input:** nothing from earlier steps. The text of all six files is written inside the code
cell itself, so you do not need to download anything.

**How it works:** the code makes a folder called `data/` and writes each of the six files into
it. If a file is already there, it is left untouched.

**Output:** one line per file showing where it is (for example `wrote data/file_events.csv`),
then the full location of the folder. The line says `wrote` even when the file was already
there; your existing copy is still not changed. The six files are:

| File | What it records |
|---|---|
| `artifact_manifest.json` | The case summary: case number, phone model, how and when the phone was copied, and the time window under investigation. |
| `file_events.csv` | When files on the phone were created, opened, copied, or deleted, with each file's **hash** (a digital fingerprint: identical files have identical hashes). |
| `app_db_messages.csv` | Short excerpts of messages sent or received in apps (Telegram, Gmail, Slack). |
| `network_events.csv` | Which internet services the phone contacted, and how much data it sent. |
| `location_events.csv` | Where the phone was, from GPS, Wi-Fi, or the mobile network. |
| `chain_of_custody.csv` | Who handled the evidence and when: the **chain of custody**. |

A `.csv` file is a simple table saved as text, one row per line. Step A.1 reads these files.

In [ ]:
# Step 0.3: write the six case files into a folder called data/, so no download is needed.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

# ARTIFACTS (the long line below) is a dictionary: a list of name -> value pairs. Each name is a
# file name, and each value is the full text of that file. You do not need to read it here.
ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "DF-2026-017",\n  "device": {\n    "platform": "Android",\n    "model": "Pixel 7",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.64",\n    "acquired_at_utc": "2026-02-18T14:32:11Z",\n    "original_image_sha256": "a5f30f7f6a6c95c17117d4ea03f2a618f9380ca379f6f31df96ab53ac49f58a8"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-02-14T20:00:00Z",\n    "end": "2026-02-16T04:00:00Z"\n  },\n  "note": "Synthetic classroom dataset for reflection-pattern demo"\n}\n', 'file_events.csv': 'timestamp_utc,path,event_type,sha256\n2026-02-15T01:05:22Z,/storage/emulated/0/Download/customers_q1.csv,created,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n2026-02-15T01:06:12Z,/storage/emulated/0/Download/customers_q1.csv,accessed,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n2026-02-15T01:07:41Z,/storage/emulated/0/Android/data/com.telegram.messenger/files/Telegram/Documents/customers_q1.csv,copied,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n2026-02-15T01:08:59Z,/storage/emulated/0/DCIM/Camera/IMG_20260215_010859.jpg,created,2a427f455903f62185111b45e22f4f55e7e5a3f7ed4d90dd357b95a8619a185d\n2026-02-15T01:12:04Z,/storage/emulated/0/Download/customers_q1.csv,deleted,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n', 'app_db_messages.csv': 'timestamp_utc,app,contact,direction,text_excerpt\n2026-02-15T01:07:58Z,Telegram,@external_consult,OUT,"sending that sheet now"\n2026-02-15T01:08:15Z,Telegram,@external_consult,OUT,"delete after download"\n2026-02-15T01:09:44Z,Gmail,personal_mail,OUT,"see attached export"\n2026-02-15T01:11:51Z,Slack,#sales-ops,IN,"Need customer list for Monday forecast"\n', 'network_events.csv': 'timestamp_utc,domain,ip,bytes_out,process\n2026-02-15T01:07:54Z,api.telegram.org,149.154.167.220,285901,org.telegram.messenger\n2026-02-15T01:09:39Z,smtp.gmail.com,142.250.114.108,194221,com.google.android.gm\n2026-02-15T01:10:08Z,drive.google.com,142.250.191.142,16992,com.android.chrome\n2026-02-15T01:11:02Z,intranet.corp.local,10.40.22.18,8422,com.microsoft.teams\n', 'location_events.csv': 'timestamp_utc,lat,lon,source\n2026-02-15T00:58:00Z,40.71274,-74.00595,gps\n2026-02-15T01:06:00Z,40.71275,-74.00593,wifi\n2026-02-15T01:12:00Z,40.71273,-74.00590,cell\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-02-18T14:32:11Z,Investigator_A,acquired_image,"UFED acquisition to encrypted evidence drive"\n2026-02-18T15:04:29Z,Investigator_A,computed_hash,"SHA-256 verified against acquisition log"\n2026-02-19T09:10:03Z,Analyst_B,created_working_copy,"Mounted read-only image; exported SQLite and CSV artifacts"\n'}

# Make the data/ folder. exist_ok=True means: no error if the folder is already there.
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
# A loop: repeat the indented lines once for each file in ARTIFACTS.
# Write the file only if it does not exist yet (so nothing is overwritten), then print its place.
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))
print('\nData folder ready:', data_dir.resolve())

## The Case in One Minute

**Case `DF-2026-017`:** a company Google Pixel 7 phone is examined after a possible leak of
customer data, shortly before an employee left the company. During the time window under
investigation, the file `customers_q1.csv` appears in the Downloads folder, is opened, is copied
into Telegram's storage, and is then deleted. There is Telegram, Gmail, and network activity
around the same time.

All times are in **UTC**, a world-standard clock. Forensic tools record times in UTC so that
evidence from different places and devices lines up.

| UTC time | What the artifacts record |
|---|---|
| 01:05:22 | `customers_q1.csv` created in Downloads |
| 01:06:12 | same file accessed (opened) |
| 01:07:41 | a file with the same hash copied into Telegram's `Documents` folder |
| 01:07:54 | outgoing internet traffic to `api.telegram.org` |
| 01:07:58 | Telegram message out: "sending that sheet now" |
| 01:08:15 | Telegram message out: "delete after download" |
| 01:09:39 | outgoing email traffic through Gmail's mail server (`smtp.gmail.com`) |
| 01:09:44 | Gmail message out: "see attached export" |
| 01:12:04 | `customers_q1.csv` deleted from Downloads |

The files also hold events that do not fit one simple story, for example a camera photo
created at 01:08:59, a Slack message **received** at 01:11:51 asking for the customer list for a
Monday forecast, and traffic to Google Drive and to a company intranet. A careful report has to
take these into account too.

These events are **consistent with** an attempt to share the file. They do **not**, by
themselves, prove that the file reached anyone. Keeping that line clear is the whole point of
this lab. In forensic terms: an **observation** is what the evidence directly shows, an
**inference** is a careful conclusion drawn from it, and an **overclaim** is a conclusion the
evidence does not support.

## Part A: Build the Reflection Loop by Hand

Think of an examination office. A junior examiner writes a draft report, a senior reviewer
marks it up, and the junior examiner fixes it. In Part A the AI plays both roles, and you run
each move yourself so you can see exactly what goes in and what comes out:

1. **Generate:** the AI analyst writes a first draft (Steps A.1 to A.3).
2. **Reflect:** a separate AI reviewer critiques the draft (Step A.4).
3. **Revise:** the analyst rewrites the draft using the critique (Step A.5).

### Step A.1: Pack the evidence into one block of text

**Purpose:** gather the evidence into a single package that we can hand to the AI, the way
you would hand a case folder to a colleague.

**Input:** the six files in the `data/` folder, written in Step 0.3.

**How it works:** the code opens each file. It copies the case summary file in full, and the
first 8 lines of each of the other five files (every file in this case is 8 lines or shorter,
so nothing is cut off). It labels each piece with its file name and joins everything into one
block of text. Then it prints the start of that block so you can check it.

**Output:** the first 600 characters of the evidence package, ending in `...`. The full package
is saved under the name `artifact_bundle`. Step A.2 attaches it to the AI's task.

In [ ]:
# Step A.1: pack the evidence files into one block of text, ready to hand to the AI model.
from pathlib import Path
from IPython.display import display_markdown  # shows the AI's reports as neat formatted text

data_dir = Path('data')  # the folder of case files written in Step 0.3

# A small helper: open one file and keep only its first 8 lines, so the AI gets a short,
# readable sample. (Every file in this case is 8 lines or shorter, so all of it is kept.)
def excerpt(filename, n_lines=8):
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])

# Build the package: each file's name, followed by its contents. The case summary
# (artifact_manifest.json) goes in whole; the five tables go in through excerpt() above.
artifact_bundle = f'''
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

file_events.csv
{excerpt('file_events.csv')}

app_db_messages.csv
{excerpt('app_db_messages.csv')}

network_events.csv
{excerpt('network_events.csv')}

location_events.csv
{excerpt('location_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
'''
# Show the first 600 characters, so you can see what the AI will receive.
print(artifact_bundle[:600], '...')

### Step A.2: Give the AI analyst its role and its task

**Purpose:** tell the AI what job it is doing and what the report must contain. The
instructions we give an AI are called a **prompt**.

**Input:** `artifact_bundle`, the evidence package from Step A.1.

**How it works:** a conversation with an AI model is kept as a list of messages. We start the
analyst's conversation with two messages. Nothing is sent to the AI yet.

- The **system** message is the job description: act as a forensic analyst, use only the
  evidence given, cite the time and the source file for every claim, and keep what the
  evidence shows (observation) separate from what you conclude from it (inference).
- The **user** message is the request: a timeline, a main explanation (the primary
  **hypothesis**), two other possible explanations, a confidence level (low, medium, or high)
  for each claim, and an evidence map. The evidence package is attached at the end.

**Output:** the message `Generator primed with 2 messages.` ("primed" means "ready"). The
conversation is saved as `generation_chat_history`; "generation" here just means "writing
the report". Step A.3 sends it to the AI.

In [ ]:
# Step A.2: write the analyst's instructions and the task. Nothing is sent to the AI yet.
# A conversation is a list (the square brackets) of messages. Each message is a dictionary
# with a 'role' (who is speaking) and its 'content' (what they say).
generation_chat_history = [
    # Message 1, role 'system': the job description the AI must follow.
    {
        'role': 'system',
        'content': (
            'You are a digital forensics analyst writing a defensible preliminary incident report. '
            'Use only the provided artifacts. Cite concrete evidence in each claim (timestamp plus '
            'artifact source). Separate observation from inference. If the user provides critique, '
            'respond with a revised version of your previous report.'
        ),
    },
    # Message 2, role 'user': the task, with the evidence package from Step A.1 attached.
    {
        'role': 'user',
        'content': (
            'Analyze this mobile forensic case and produce a preliminary incident report.\n'
            'Return: (1) a timeline, (2) a primary hypothesis, (3) two alternative hypotheses, '
            '(4) a confidence level (low / medium / high) per claim, (5) an explicit evidence map.\n\n'
            f'Artifacts:\n{artifact_bundle}'
        ),
    },
]
print('Generator primed with', len(generation_chat_history), 'messages.')

### Step A.3: Get the first draft report

**Purpose:** get the AI's first draft, the way a junior examiner hands in a first attempt.

**Input:** `generation_chat_history` from Step A.2, plus `client` and `MODEL` from Step 0.2.

**How it works:** the code sends the whole conversation to the AI model and waits for its
reply, which can take a little while. It saves the reply, adds it to the analyst's
conversation (so the AI will remember its own draft when it revises in Step A.5), and shows it
as formatted text.

**Output:** the first draft report on screen. It will probably sound confident. As you read,
note any claim that goes further than the evidence: for example, saying someone **received**
the file, when the evidence only shows outgoing activity. The draft is saved as
`incident_report_draft`. Step A.4 hands it to a reviewer.

In [ ]:
# Step A.3: send the conversation to the AI model and get its first draft report.
# client.chat.completions.create(...) sends the messages and waits for the reply.
# .choices[0].message.content then picks out just the text of that reply.
incident_report_draft = client.chat.completions.create(
    model=MODEL,
    messages=generation_chat_history,
).choices[0].message.content

# Add the draft to the analyst's conversation as the AI's own reply (role 'assistant'),
# so it remembers what it wrote when it is asked to revise in Step A.5.
generation_chat_history.append({'role': 'assistant', 'content': incident_report_draft})
# Show the draft as formatted text.
display_markdown(incident_report_draft, raw=True)

### Step A.4: Reflect: have a reviewer critique the draft

**Purpose:** get a second opinion. A separate AI **reviewer** checks the draft for
overclaiming (saying more than the evidence supports), claims with no source, problems with
times, and chain-of-custody gaps.

**Input:** `incident_report_draft` from Step A.3.

**How it works:** the code starts a new, separate conversation. Its system message makes the AI
a senior forensic reviewer: for each problem, say why it is a risk and how to fix it, and if
there are no real problems, reply with exactly `[[OK]]`. The user message is the draft. The
code sends this to the AI and shows the reply.

Note: the reviewer sees **only the draft report**, not the evidence files. So it judges whether
claims are properly supported and worded, not whether the files say something else. Checking
the report against the files is still your job.

**Output:** the reviewer's critique on screen, saved as `critique`. If you see `[[OK]]`, the
reviewer found nothing important to fix. Step A.5 passes the critique back to the analyst.

In [ ]:
# Step A.4: set up a separate AI 'reviewer' and ask it to critique the draft from Step A.3.
# This is a brand-new conversation. The reviewer sees only the draft, not the evidence files.
reflection_chat_history = [
    # Message 1, role 'system': the reviewer's job description.
    {
        'role': 'system',
        'content': (
            'You are a senior digital forensics reviewer. Critique the report for evidentiary rigor. '
            'For each issue, say why it is a risk and how to fix it. Focus on provenance gaps, '
            'timestamp inconsistencies, chain-of-custody concerns, and overclaiming (for example '
            'claiming delivery or receipt that the artifacts do not confirm). '
            'If there are no material issues, output exactly [[OK]].'
        ),
    },
    # Message 2, role 'user': the draft to review.
    {'role': 'user', 'content': f'Please review this forensic report:\n\n{incident_report_draft}'},
]

# Send the reviewer's conversation to the AI model, keep the text of its reply, and show it.
critique = client.chat.completions.create(
    model=MODEL,
    messages=reflection_chat_history,
).choices[0].message.content
display_markdown(critique, raw=True)

### Step A.5: Revise: hand the critique back to the analyst

**Purpose:** let the analyst fix its report, the way a junior examiner revises after a
supervisor's comments.

**Input:** `critique` from Step A.4, and the analyst's conversation `generation_chat_history`,
which already holds the instructions, the task, the evidence, and the first draft (from Steps
A.2 and A.3).

**How it works:** the code adds one more message to the analyst's conversation: "revise your
previous report using this critique", with the critique attached. Then it sends the
conversation to the AI and shows the reply.

**Output:** the revised report on screen, saved as `revised_incident_report`. Compare it with
the first draft: are the claims more careful? Does each one cite a time and a source? You will
use both versions in Your Evidence Check.

In [ ]:
# Step A.5: give the critique to the analyst and ask for a revised report.
# Add the critique to the analyst's conversation as a new request.
generation_chat_history.append({
    'role': 'user',
    'content': f'Revise your previous forensic report using this critique.\n\nCritique:\n{critique}',
})

# Send the whole conversation (instructions, task, draft, critique) and show the revised report.
revised_incident_report = client.chat.completions.create(
    model=MODEL,
    messages=generation_chat_history,
).choices[0].message.content
display_markdown(revised_incident_report, raw=True)

### Step A.6: Repeat when needed

This step has no code; it just explains what comes next.

In a full loop, the revised report from Step A.5 would go back to the reviewer, then back to
the analyst, and so on. The loop stops when the reviewer replies `[[OK]]`, or when a set number
of rounds is reached, so it can never run forever. Doing that by hand is slow, so Part B does
the repeating for you.

## Part B: Run the Same Workflow with a Reusable `ReflectionAgent`

In Part A you did each move yourself. Part B packs the exact same loop (draft, review, revise,
repeat) into one ready-made helper called `ReflectionAgent`, so you can run it with a single
command. An **agent** is a program that uses an AI model to carry out several steps toward a
goal on its own.

The helper below is a self-contained copy adapted from The Neural Maze's
agentic-patterns-course (MIT License).

### Step B.1: Load the ready-made agent

**Purpose:** load the `ReflectionAgent` helper so that Step B.2 can use it.

**Input:** nothing from earlier steps. All the code is inside the cell.

**How it works:** the cell only **defines** the agent (it teaches Python what a
`ReflectionAgent` is); it does not run it yet. You do not need to read every line. To see the
link with Part A, look at the `run()` part near the bottom. It writes a draft, asks the
reviewer, stops if the reply contains `[[OK]]`, and otherwise passes the critique back for a
revision. To keep each conversation short, it lets older drafts and critiques drop out, but it
always keeps the instructions and the original task.

**Output:** nothing is printed. `ReflectionAgent` is now ready for Step B.2.

In [ ]:
# Step B.1: the packaged Reflection Pattern. Same idea as Part A, wrapped up so it can be reused.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

# This is the toolbox behind ReflectionAgent. Skim the names; you do not need to follow every
# line. The part that matters most is ReflectionAgent.run() at the very bottom: it is the same
# draft -> review -> revise loop you ran by hand in Part A.


# ----------------------------------------------------------------- small helpers for messages
# completions_create: send a list of messages to the AI model and return the text of its reply.
def completions_create(client, messages, model):
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# build_prompt_structure: wrap a piece of text as one message with a role: 'system' (the job
# description), 'user' (a request), or 'assistant' (something the AI said).
def build_prompt_structure(prompt, role, tag=""):
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


# update_chat_history: add one new message to the end of a conversation.
def update_chat_history(history, msg, role):
    history.append(build_prompt_structure(prompt=msg, role=role))


# ChatHistory: a conversation (a list of messages) with an optional size limit. When the list is
# full, the oldest message is dropped to make room for the new one. -1 means "no limit".
class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        if messages is None:
            messages = []
        super().__init__(messages)
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)

    # show_messages: print a short preview of every message, handy for checking what the AI saw.
    def show_messages(self):
        if not self:
            print("ChatHistory is empty.")
            return
        for idx, message in enumerate(self, start=1):
            if isinstance(message, dict):
                role = message.get("role", "unknown")
                content = message.get("content", "")
                preview = content if len(content) < 160 else content[:157] + "..."
                print(f"{idx}. [{role}] {preview}")
            else:
                print(f"{idx}. {message}")


# FixedPrefixChatHistory: like ChatHistory, but the first few messages (for example the
# instructions and the original task) are never dropped. Only newer messages rotate out.
class FixedPrefixChatHistory(ChatHistory):
    """Keeps an initial prefix pinned (system + original task) while newer messages rotate."""

    def __init__(self, messages=None, total_length=-1, pinned_prefix_len=1):
        super().__init__(messages, total_length)
        self.pinned_prefix_len = pinned_prefix_len

    def append(self, msg):
        # When full, drop the oldest message that comes AFTER the pinned ones.
        if self.total_length != -1 and len(self) == self.total_length:
            self.pop(self.pinned_prefix_len)
        list.append(self, msg)


# ----------------------------------------------------------------- the agent
# General instructions the agent adds to whatever role instructions you give it.
# For the writer (analyst): always hand back a full, revised version after a critique.
BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

# For the reviewer: list problems and fixes, or reply exactly [[OK]] when there are none.
BASE_REFLECTION_SYSTEM_PROMPT = """
You are tasked with generating critique and recommendations to the user's generated content.
If the user content has something wrong or something to be improved, output a list of recommendations
and critiques. If the user content is correct and there's no recommendations and critiques, you should
output exactly the string: [[OK]], otherwise, you must output [[not OK]].
"""


class ReflectionAgent:
    """Generates a draft, critiques it, and revises it, looping until the critique says [[OK]]."""

    # Set up the agent: remember which AI connection (client) and which model to use.
    def __init__(self, model="openai/gpt-oss-120b", client=None):
        if client is None:
            raise ValueError(
                "ReflectionAgent requires a `client` instance that supports "
                "`chat.completions.create(...)`."
            )
        self.client = client
        self.model = model

    # Send one conversation to the AI model. If verbose is on, print the reply under a title.
    def _request_completion(self, history, verbose=0, log_title="COMPLETION"):
        output = completions_create(self.client, history, self.model)
        if verbose > 0:
            print(f"\n\n{log_title}\n\n{output}")
        return output

    # generate: the analyst writes (or rewrites) the report.
    def generate(self, generation_history, verbose=0):
        return self._request_completion(generation_history, verbose, log_title="GENERATION")

    # reflect: the reviewer critiques the latest report.
    def reflect(self, reflection_history, verbose=0):
        return self._request_completion(reflection_history, verbose, log_title="REFLECTION")

    # run: the whole loop. This is Part A's draft -> review -> revise, repeated up to n_steps times.
    def run(self, user_msg, generation_system_prompt="", reflection_system_prompt="",
            n_steps=3, verbose=0):
        # Add the general instructions (above) to the role instructions you passed in.
        generation_system_prompt += BASE_GENERATION_SYSTEM_PROMPT
        reflection_system_prompt += BASE_REFLECTION_SYSTEM_PROMPT

        # The analyst's conversation: its instructions and the task, then the latest draft and
        # the latest critique.
        generation_history = FixedPrefixChatHistory(
            [
                build_prompt_structure(prompt=generation_system_prompt, role="system"),
                build_prompt_structure(prompt=user_msg, role="user"),
            ],
            total_length=4,       # at most 4 messages: instructions, task, latest draft, critique
            pinned_prefix_len=2,  # never drop the first 2 (the instructions and the task)
        )
        # The reviewer's conversation: its instructions, then the newest drafts and critiques.
        # Note that the reviewer is never shown the evidence itself, only the drafts.
        reflection_history = FixedPrefixChatHistory(
            [build_prompt_structure(prompt=reflection_system_prompt, role="system")],
            total_length=4,
            pinned_prefix_len=1,
        )

        generation = ""
        # A loop: repeat the indented lines once per round, up to n_steps rounds.
        for step in range(n_steps):
            if verbose > 0:
                print(f"\n{'=' * 50}\nSTEP {step + 1}/{n_steps}\n{'=' * 50}")

            # 1. The analyst writes (or rewrites) the report. Save it in the analyst's memory and
            #    hand it to the reviewer.
            generation = self.generate(generation_history, verbose=verbose)
            update_chat_history(generation_history, generation, "assistant")
            update_chat_history(reflection_history, generation, "user")

            # 2. The reviewer critiques it.
            critique = self.reflect(reflection_history, verbose=verbose)

            # 3. If the critique contains [[OK]], the reviewer is satisfied: stop the loop.
            if "[[OK]]" in critique:
                update_chat_history(reflection_history, critique, "assistant")
                print("\nStop sequence found. Stopping the reflection loop.\n")
                break

            # Otherwise, pass the critique to the analyst as the next request, and go round again.
            update_chat_history(reflection_history, critique, "assistant")
            update_chat_history(generation_history, critique, "user")

        # Hand back the last report the analyst wrote.
        return generation

### Step B.2: Create the agent and run it

**Purpose:** run the whole reflection loop automatically on the same case, with no hand steps.

**Input:** `ReflectionAgent` from Step B.1, `artifact_bundle` (the evidence package) from Step
A.1, and `client` and `MODEL` from Step 0.2.

**How it works:** the code creates the agent, writes short instructions for the analyst and
the reviewer (shorter versions of Part A's), and gives it the same task and evidence. The agent
also adds its own general instructions; for example, it tells the reviewer to reply `[[OK]]`
when satisfied or `[[not OK]]` when not. Then it repeats draft, review, revise on its own. It
stops when the reviewer's reply contains `[[OK]]`, or after 3 rounds (`n_steps=3`), which also
keeps a free key within its limits.

**Output:** a printed log. Each round shows `STEP 1/3` (then 2/3, 3/3), the analyst's
`GENERATION` (its report), and the reviewer's `REFLECTION` (its critique). If the reviewer is
satisfied you will see `Stop sequence found. Stopping the reflection loop.` The last report
written is saved as `final_report`, which Step B.3 shows. If all 3 rounds run without an
`[[OK]]`, the final critique appears in the log but was never applied to the report: read it.

In [ ]:
# Step B.2: run the whole draft -> review -> revise loop automatically on the same case.
# Create the agent, giving it the AI model name and the connection from Step 0.2.
agent = ReflectionAgent(model=MODEL, client=client)

# Short instructions (prompts) for the two roles: the analyst who writes, the reviewer who checks.
generation_system_prompt = (
    'You are a digital forensics analyst writing a defensible incident report. '
    'Use only user-provided evidence, cite timestamps and artifact sources, and revise when '
    'critique is provided.'
)
reflection_system_prompt = (
    'You are a senior digital forensics reviewer. Flag overclaiming, provenance gaps, and weak '
    'uncertainty language. Output exactly [[OK]] when there is nothing material to improve.'
)

# Start the loop. user_msg is the task plus the evidence package from Step A.1.
# n_steps=3: run at most 3 rounds. verbose=1: print every draft and critique so you can watch.
final_report = agent.run(
    user_msg=(
        'Analyze this mobile forensic case and produce a preliminary incident report.\n'
        'Return: (1) a timeline, (2) a primary hypothesis, (3) two alternative hypotheses, '
        '(4) a confidence level per claim, (5) an explicit evidence map.\n\n'
        f'Artifacts:\n{artifact_bundle}'
    ),
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=3,
    verbose=1,
)

### Step B.3: Read the final report

**Purpose:** read the report the loop ended with, as one clean document.

**Input:** `final_report` from Step B.2.

**How it works:** the code shows the report as formatted text.

**Output:** the final report on screen. Compare it with the first draft (Step A.3) and the
revised report (Step A.5). You will use all of them in Your Evidence Check.

In [ ]:
# Step B.3: show the final report from Step B.2 as formatted text.
display_markdown(final_report, raw=True)

## Your Evidence Check

**Purpose:** show that you, the examiner, can judge the AI's work against the evidence.

**Input:** the first draft (Step A.3), the critique (Step A.4), the revised report (Step A.5),
the Part B output (Steps B.2 and B.3), and the case files (Step 0.3 and The Case in One
Minute).

**How it works:** in the cell below, replace each `... your answer ...` text with your own
answer, keeping the quote marks around it, then run the cell. Use your own words. Base every
answer on the artifacts and the outputs above, not on general knowledge. Tip: do not use the `'`
character inside an answer (write "do not", not "don't"), or Python will stop with an error.

**Output:** the cell prints your four answers back, so you can check them before you submit.

**Questions:**

1. Find one claim in the **first draft** that was too strong. Which artifacts support a narrower
   claim instead?
2. Pick one claim in the **revised report**. Give the artifact source and UTC timestamp that back
   it, then state one limit on that claim.
3. Did the critique improve the report's **uncertainty language** (the words that show how sure
   the report is, such as "may", "likely", or "not confirmed")? Quote one phrase that
   changed from draft to revision.
4. What forensic judgment must **stay with a human**, even after the loop ends?

In [ ]:
# Your Evidence Check: replace each '... ...' text with your answer, then run this cell.
answers = {
    'q1_overclaim': '... your answer ...',
    'q2_supported_claim': '... artifact + timestamp + limit ...',
    'q3_uncertainty_change': '... phrase before -> after ...',
    'q4_human_judgment': '... your answer ...',
}
# Print each question label with your answer next to it.
for k, v in answers.items():
    print(k, '->', v)

## Submit

Save the notebook with the draft, critique, revised report, Part B output, and your four answers
filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.
- Hand in the `.ipynb` file as your instructor directs.

---

*Case data and the agentic-pattern course structure are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). Agent code is*
*adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*